In [1]:

import random
from datetime import datetime, timedelta

from faker import Faker
from pyspark.sql import SparkSession

In [2]:
Faker.seed(42)
fake = Faker(['ko_KR', 'en_US'])

In [3]:
def random_date(days=30):
    return datetime.now() - timedelta(days=random.randint(0, days))


def fake_natural_product_name():
    pattern = random.choice([
        lambda: fake.catch_phrase(),
        lambda: fake.bs().title(),
        lambda: f"{fake.color_name()} {fake.word().title()} Edition",
        lambda: f"{fake.word().title()} {fake.word().title()} Series",
        lambda: fake.sentence(nb_words=random.randint(3, 6)).replace(".", "")
    ])
    return pattern()


def generate_orders(count: int, user_id_range: tuple[int, int] = (1, 100), quantity_range: tuple[int, int] = (1, 5), unit_price_range: tuple[int, int] = (10_000, 500_000)) -> list[dict]:
    categories = ["전자기기", "가전", "패션", "도서", "생활용품"]
    order_statuses = ["주문완료", "배송중", "배송완료", "취소"]
    payment_methods = ["카드", "계좌이체", "카카오페이", "네이버페이"]
    orders: list[dict] = []
    for i in range(count):
        quantity = random.randint(*quantity_range)
        unit_price = random.randint(*unit_price_range)

        orders.append({
            "order_id": i + 1,
            "user_id": random.randint(*user_id_range),
            "product_name": fake_natural_product_name(),
            "category": random.choice(categories),
            "quantity": quantity,
            "unit_price": unit_price,
            "total_price": quantity * unit_price,
            "order_status": random.choice(order_statuses),
            "order_date": random_date(),
            "payment_method": random.choice(payment_methods),
            "shipping_address": fake.address(),
        })

    return orders

In [4]:
spark = SparkSession.builder.appName("S3 Example").master("spark://localhost:7077").getOrCreate()
spark.createDataFrame(data=generate_orders(100000)).write.mode("overwrite").parquet("s3a://mmix-prod-dataengineer-datalakehouse/mmix/orders/year=2026/month=01/day=09")
spark.createDataFrame(data=generate_orders(100000)).write.mode("overwrite").parquet("s3a://mmix-prod-dataengineer-datalakehouse/mmix/orders/year=2026/month=01/day=10")

26/01/09 15:05:22 WARN Utils: Your hostname, genius.local resolves to a loopback address: 127.0.0.1; using 10.12.2.51 instead (on interface en0)
26/01/09 15:05:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/09 15:05:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/09 15:05:33 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/01/09 15:05:33 WARN VersionInfoUtils: The AWS SDK for Java 1.x reached end of support on December 31, 2025. For more information, see https://aws.amazon.com/blogs/developer/the-aws-sdk-for-java-1-x-is-in-maintenance-mode-effective-july-31-2024/
You can print where on the file system the AWS SDK for Java 1.x core runtime is located by setting the 

In [5]:
spark.read.parquet("s3a://mmix-prod-dataengineer-datalakehouse/mmix/orders/year=2026/month=01/*").count()

200000